# 第13章 RAG

## 13.2 基本的なRAGのシステムの実装

### LangChainでLLMと文埋め込みモデルを使う

#### 環境の準備

In [1]:
!pip install transformers[torch,sentencepiece] langchain==0.2.12 langchain-community==0.2.11 langchain-huggingface==0.0.3 faiss-cpu jq

In [2]:
from huggingface_hub import notebook_login

# Hugging Face Hubにログイン
notebook_login()

In [3]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### LangChainでLLMを使う

In [4]:
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

# HuggingFaceHubにおけるモデル名を指定
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"

# モデルを読み込む
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# トークナイザを読み込む
tokenizer = AutoTokenizer.from_pretrained(model_name)

# テキスト生成用のパラメータを指定
generation_config = {
    "max_new_tokens": 128,
    "do_sample": False,
    "temperture": None,
    "top_p": None,
}

# テキスト生成を行うパイプラインを作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

# パイプラインからLangChainのLLMコンポーネントを作成
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
from pprint import pprint

# modelに入力する会話データ
llm_prompt_messages = [
    {"role": "user", "content": "四国地方で一番高い山は？"},
]

# 会話データにチャットテンプレートを適用し、内容を確認
llm_prompt_text = tokenizer.apply_chat_template(
    llm_prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
pprint(llm_prompt_text)


'<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：'


In [6]:
# LLMへの入力を実行し結果を確認
llm_output_message = llm.invoke(llm_prompt_text)
pprint(llm_output_message)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：四国地方で一番高い山は、徳島県と高知県の県境にある剣山で、標高は1,955メートルです。'


#### Chat Modelコンポーネントの利用

In [7]:
from langchain_huggingface import ChatHuggingFace

# LLMコンポーネントからChatModelコンポーネントを作成
chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

# Chat Modelに入力する会話データ
chat_messages = [HumanMessage(content="四国地方で一番高い山は？")]
#print(chat_messages)

# Chat Modelによるチャットテンプレート適用後の入力文字列をかくにん
chat_prompt = chat_model._to_chat_prompt(chat_messages)
print(chat_prompt)

<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：


In [9]:
# Chat Modelに会話データを入力し、出力を確認
chat_output_message = chat_model.invoke(chat_messages)
pprint(chat_output_message)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AIMessage(content='<s>ユーザ：四国地方で一番高い山は？</s><s>アシスタント：四国地方で一番高い山は、徳島県と高知県の県境にある剣山で、標高は1,955メートルです。', id='run-a0cd86e5-f8ba-4011-897f-f9afe9d24078-0')


In [10]:
# Chat Modelが出力したテキストからモデルの応答部分のみを抽出
response_text = chat_output_message.content[len(chat_prompt) :]
print(response_text)

四国地方で一番高い山は、徳島県と高知県の県境にある剣山で、標高は1,955メートルです。


#### Chainを構築する

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# 任意のqueryからプロンプトを構築するPrompt Templateを作成
prompt_template = ChatPromptTemplate.from_messages(
    [("user", "{query}")]
)

# Prompt Templateを実行し、結果を確認
prompt_template_output = prompt_template.invoke(
    {"query": "四国地方で一番高い山は？"}
)
pprint(prompt_template_output)

ChatPromptValue(messages=[HumanMessage(content='四国地方で一番高い山は？')])


In [13]:
# Prompt TemplateとChat Modelを連結したChainを作成
chain = prompt_template | chat_model
print(chain)

# Chainをジック押し結果を確認
chain_output = chain.invoke({"query": "四国地方で一番高い山は？"})
pprint(chain_output)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


first=ChatPromptTemplate(input_variables=['query'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], template='{query}'))]) last=ChatHuggingFace(llm=HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}), tokenizer=LlamaTokenizer(name_or_path='llm-book/Swallow-7b-hf-oasst1-21k-ja', vocab_size=43176, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}), mod